In [25]:
!pip install -q geemap earthengine-api geopandas pandas numpy

In [26]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

In [27]:
ee.Authenticate()
ee.Initialize()

In [28]:
ee.Initialize(project="solid-garden-458417-t4")

In [29]:
khulna = gpd.read_file("khulna_boundary_wgs84.geojson")

grid_morphology = gpd.read_file(
    "khulna_morphology.geojson"
)

In [30]:
khulna_ee = geemap.geopandas_to_ee(khulna)
grid_ee = geemap.geopandas_to_ee(grid_morphology)

In [31]:
viirs = (
    ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG")
    .select("avg_rad")
)

In [32]:
def annual_viirs(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)

    image = (
        viirs
        .filterDate(start, end)
        .mean()
        .clip(khulna_ee)
        .rename(f"ntl_{year}")
        .set("year", year)
    )

    return image

In [33]:
ntl_2024 = annual_viirs(2024)

In [34]:
Map = geemap.Map()

Map.addLayer(
    ntl_2024,
    {"min": 0, "max": 40},
    "VIIRS NTL 2024"
)

Map.addLayer(khulna_ee, {}, "Khulna Boundary")
Map.centerObject(khulna_ee, 10)

Map

Map(center=[22.364757409609016, 89.45273382891959], controls=(WidgetControl(options=['position', 'transparent_…

In [35]:
vis_params = {
    "min": 0,
    "max": 15,
    "palette": [
        "black",
        "purple",
        "blue",
        "cyan",
        "yellow",
        "red"
    ]
}

In [36]:
Map = geemap.Map()

Map.addLayer(
    ntl_2024,
    vis_params,
    "VIIRS 2024"
)

Map.addLayer(
    khulna_ee,
    {},
    "Khulna Boundary"
)

Map.centerObject(khulna_ee, 10)

Map

Map(center=[22.364757409609016, 89.45273382891959], controls=(WidgetControl(options=['position', 'transparent_…

In [37]:
ntl_2024 = ntl_2024.updateMask(
    ntl_2024.gt(0)
)

In [38]:
Map = geemap.Map()

Map.addLayer(
    ntl_2024,
    vis_params,
    "VIIRS 2024"
)

Map.addLayer(
    khulna_ee,
    {},
    "Khulna Boundary"
)

Map.centerObject(khulna_ee, 10)

Map

Map(center=[22.364757409608934, 89.45273382891959], controls=(WidgetControl(options=['position', 'transparent_…

In [39]:
years = list(range(2012, 2025))

annual_ntl = {
    year: annual_viirs(year)
    for year in years
}

In [40]:
ntl_2012 = annual_ntl[2012]
ntl_2024 = annual_ntl[2024]

ntl_change = (
    ntl_2024
    .subtract(ntl_2012)
    .rename("ntl_change_2012_2024")
)

In [41]:
Map = geemap.Map()

Map.addLayer(
    ntl_change,
    {
        "min": -5,
        "max": 10,
        "palette": ["blue", "black", "yellow", "red"]
    },
    "NTL Change 2012-2024"
)

Map.addLayer(khulna_ee, {}, "Khulna Boundary")
Map.centerObject(khulna_ee, 10)

Map

Map(center=[22.364757409609016, 89.45273382891959], controls=(WidgetControl(options=['position', 'transparent_…